In [ ]:
import tensorflow as tf
import warnings
warnings.filterwarnings("ignore")
from tensorflow import keras
#import tensorflow_datasets as tfds
import numpy as np
import pandas as pd
import scipy.io as sio
from tensorflow.keras.preprocessing.image import ImageDataGenerator

#Oxford 102 datasets

image_classes = sio.loadmat("imagelabels.mat")
classes = image_classes['labels']

set_id = sio.loadmat("setid.mat")

train_id = set_id['tstid']
test_id = set_id['trnid']
val_id = set_id['valid']

print(train_id.shape)

train_class = np.array([classes[0][i-1] for i in train_id[0]])
test_class = np.array([classes[0][i-1] for i in test_id[0]])
val_class = np.array([classes[0][i-1] for i in val_id[0]])

def files(ids):
    list1 = []
    prefix = ''
    beginning = 'image_'
    extension = '.jpg'
    for i in ids:
        if i > 999:
            prefix = '0'
        elif i > 99:
            prefix = '00'
        elif i > 9:
            prefix = '000'
        elif i > 0:
            prefix = '0000'
        else:
            continue
        filename = beginning + prefix + str(i) + extension
        list1.append(filename)
    return list1

train_file = files(train_id[0])
test_file = files(test_id[0])
val_file = files(val_id[0])
print(len(train_file), len(test_file), len(val_file))

train_data = np.array([[train_file[i], train_class[i]] for i in range(len(train_file))])
test_data = np.array([[test_file[i], test_class[i]] for i in range(len(test_file))])
val_data = np.array([[val_file[i], val_class[i]] for i in range(len(val_file))])

df_train = pd.DataFrame(train_data, columns = ['filename', 'class'])
df_test = pd.DataFrame(test_data, columns = ['filename', 'class'])
df_val = pd.DataFrame(val_data, columns = ['filename', 'class'])
print(df_train['class'])

idg = ImageDataGenerator(rescale = 1./255.)
class_labels = [str(x) for x in range(102)]

BatchSize = 50

train_ds = idg.flow_from_dataframe(dataframe = df_train, directory = "102flowers.tar/jpg/", classes = class_labels, batch_size = BatchSize)
test_ds = idg.flow_from_dataframe(dataframe = df_test, directory = "102flowers.tar/jpg/", classes = class_labels, batch_size = BatchSize)
val_ds = idg.flow_from_dataframe(dataframe = df_val, directory = "102flowers.tar/jpg/", classes = class_labels, batch_size = BatchSize)


(1, 6149)
6149 1020 1020
0         1
1         1
2         1
3         1
4         1
       ... 
6144    102
6145    102
6146    102
6147    102
6148    102
Name: class, Length: 6149, dtype: object
Found 0 validated image filenames belonging to 102 classes.
Found 0 validated image filenames belonging to 102 classes.
Found 0 validated image filenames belonging to 102 classes.


In [23]:
Nepochs = 90
DropoutValue = 0.5

xpix = train_ds[0][0][0].shape[0]
ypix = train_ds[0][0][0].shape[1]
zpix = train_ds[0][0][0].shape[2]


model = keras.models.Sequential()
model.add(keras.layers.Conv2D(128, (3, 3), activation = 'relu', input_shape = (xpix, ypix, zpix)))
model.add(keras.layers.MaxPooling2D(2, 2))

model.add(keras.layers.Conv2D(128, (3, 3), activation = 'relu'))
model.add(keras.layers.Conv2D(128, (3, 3), activation = 'relu'))
model.add(keras.layers.Conv2D(128, (3, 3), activation = 'relu'))
model.add(keras.layers.MaxPooling2D(2, 2))

model.add(keras.layers.Conv2D(64, (3, 3), activation = 'relu'))
model.add(keras.layers.Conv2D(64, (3, 3), activation = 'relu'))
model.add(keras.layers.Conv2D(64, (3, 3), activation = 'relu'))
model.add(keras.layers.MaxPooling2D(2, 2))

model.add(keras.layers.Conv2D(64, (3, 3), activation = 'relu'))
model.add(keras.layers.Conv2D(64, (3, 3), activation = 'relu'))
model.add(keras.layers.Conv2D(64, (3, 3), activation = 'relu'))
model.add(keras.layers.MaxPooling2D(2, 2))

model.add(keras.layers.Conv2D(32, (3, 3), activation = 'relu'))
model.add(keras.layers.Conv2D(32, (3, 3), activation = 'relu'))
model.add(keras.layers.Conv2D(32, (3, 3), activation = 'relu'))
model.add(keras.layers.MaxPooling2D(2, 2))

model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(64, activation = 'relu'))
model.add(keras.layers.Dropout(DropoutValue))
model.add(keras.layers.Dense(64, activation = 'relu'))
model.add(keras.layers.Dropout(DropoutValue))
model.add(keras.layers.Dense(102, activation = 'softmax'))

model.summary()

loss_fn = keras.losses.CategoricalCrossentropy()

model.compile(optimizer = 'adam', loss = loss_fn, metrics = ['accuracy'])
history = model.fit(train_ds, epochs = Nepochs, validation_data = val_ds)

ValueError: Asked to retrieve element 0, but the Sequence has length 0

In [21]:
import matplotlib.pyplot as plt

print('history keys = ', history.history.keys())

print("\n\033[1mDisplay the evolution of the accuracy as a function of the training epoch\033[0m")
print('\n N(epochs) = ', Nepochs)

print('accuracy (train) = ', history.history)
print('accuracy (test) = ', history.history['val_accuracy'])

plt.plot(history.history['accuracy'], color = 'deeppink')
plt.plot(history.history['val_accuracy'], color = 'dodgerblue')
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validate'], loc='lower right')
plt.show()
plt.clf()

print("\n\033[1mDisplay the evolution of the loss as a function of the training epoch\033[0m")
print("  N(Epochs)        = ", Nepochs)

print('loss (train) = ', history.history['loss'])
print('loss (test) = ', history.history['val_loss'])


plt.plot(history.history['loss'], color = 'deeppink')
plt.plot(history.history['val_loss'], color = 'dodgerblue')
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validate'], loc = 'upper right')
plt.show()

NameError: name 'history' is not defined

In [ ]:
predict = model.predict(test_ds[0][0])

for i in range(6):
    
    plt.imshow(test_ds[0][0][i])
    plt.show()
    print('true class = ', test_ds[0][1][i].argmax())
    print('predicted class = ', predict[i].argmax())


In [ ]:
#evaluation plots = series of bar graphs 20-30 classes each correct and incorrect. 
#+ confusion matrix = true class vs predicted class.

true = []
false = []

for i in range(len(predict)):
    print(i)
    if test_ds[0][1][i].argmax() == predict[i].argmax():
        true.append(test_ds[0][1][i].argmax())
    else:
        false.append(test_ds[0][1][i].argmax())
        
correct = []
incorrect = []

for i in range(102):
    correct.append(true.count(i))
    incorrect.append(false.count(i))
print(correct)
    
c_1, c_2, c_3, c_4, c_5 = correct[:20], correct[20:40], correct[40:60], correct[60:80], correct[80:]
i_1, i_2, i_3, i_4, i_5 = incorrect[:20], incorrect[20:40], incorrect[40:60], incorrect[60:80], incorrect[80:]

accuracy_1 = {
    'Correct1':c_1,
    'Incorrect1':i_1
}

accuracy_2 = {
    'Correct' :c_2,
    'Incorrect':i_2
}

accuracy_3 = {
    'Correct' :c_3,
    'Incorrect':i_3
}

accuracy_4 = {
    'Correct' :c_4,
    'Incorrect':i_4
}

accuracy_5 = {
    'Correct' :c_5,
    'Incorrect':i_5
}



In [ ]:
x1 = np.array(class_labels[:20], dtype = 'float64')
x2 = np.array(class_labels[20:40], dtype = 'float64')
x3 = np.array(class_labels[40:60], dtype = 'float64')
x4 = np.array(class_labels[60:80], dtype = 'float64')
x5 = np.array(class_labels[80:], dtype = 'float64')
width = 0.25  # the width of the bars
multiplier = 0

fig, ax = plt.subplots(layout = 'constrained')
for cls, num in accuracy_1.items():
    offset = width * multiplier
    rects = ax.bar(x1 + offset, num, width, label=cls)
    ax.bar_label(rects, padding=3)
    multiplier += 1
    
ax.set_ylabel('Number of Predictions')
ax.set_title('Accuracy of CNN by Class')
ax.set_xticks(x1 + width, class_labels[:20])
ax.legend(loc='upper right', ncols=3)
ax.set_ylim(0, max(correct)+2)

plt.show()

fig, ax = plt.subplots(layout = 'constrained')
for cls, num in accuracy_2.items():
    offset = width * multiplier
    rects = ax.bar(x2 + offset, num, width, label=cls)
    ax.bar_label(rects, padding=3)
    multiplier += 1
    
ax.set_ylabel('Number of Predictions')
ax.set_title('Accuracy of CNN by Class')
ax.set_xticks(x2 + width, class_labels[20:40])
ax.legend(loc='upper right', ncols=3)
ax.set_ylim(0, max(correct)+2)

plt.show()

fig, ax = plt.subplots(layout = 'constrained')
for cls, num in accuracy_3.items():
    offset = width * multiplier
    rects = ax.bar(x3 + offset, num, width, label=cls)
    ax.bar_label(rects, padding=3)
    multiplier += 1
    
ax.set_ylabel('Number of Predictions')
ax.set_title('Accuracy of CNN by Class')
ax.set_xticks(x3 + width, class_labels[40:60])
ax.legend(loc='upper right', ncols=3)
ax.set_ylim(0, max(correct)+2)

plt.show()

fig, ax = plt.subplots(layout = 'constrained')
for cls, num in accuracy_4.items():
    offset = width * multiplier
    rects = ax.bar(x4 + offset, num, width, label=cls)
    ax.bar_label(rects, padding=3)
    multiplier += 1
    
ax.set_ylabel('Number of Predictions')
ax.set_title('Accuracy of CNN by Class')
ax.set_xticks(x4 + width, class_labels[60:80])
ax.legend(loc='upper right', ncols=3)
ax.set_ylim(0, max(correct)+2)

plt.show()

fig, ax = plt.subplots(layout = 'constrained')
for cls, num in accuracy_5.items():
    offset = width * multiplier
    rects = ax.bar(x5 + offset, num, width, label=cls)
    ax.bar_label(rects, padding=3)
    multiplier += 1
    
ax.set_ylabel('Number of Predictions')
ax.set_title('Accuracy of CNN by Class')
ax.set_xticks(x5 + width, class_labels[80:])
ax.legend(loc='upper right', ncols=3)
ax.set_ylim(0, max(correct)+2)

plt.show()